# Descrption
I help you Build embedings to Chroma


In [ ]:
# @title Installs
!pip install "langchain>=0.2.7" \ "langchain-core>=0.2.6" \ "langchain-community>=0.2.6"
!pip install -qU "langchain-chroma>=0.1.2"
!pip install -qU langchain-community faiss-cpu
!pip install --upgrade langchain-google-genai
!pip install cohere
# !pip install -qU langchain-openai
# !pip install --upgrade google-genai
# !pip install chromadb

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.7 MB/s eta 0:00:00
  Attempting uninstall: httpx-sse
    Found existing installation: httpx-sse 0.4.3
    Uninstalling httpx-sse-0.4.3:
      Successfully uninstalled httpx-sse-0.4.3


In [ ]:
# @title imports
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
import os, json, hashlib, uuid, textwrap, shutil, ast
from pathlib import Path
from typing import List, Dict, Any
from google.colab import userdata
# --- LangChain / Chroma ---
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================
# @title Configure your API keys
# ============================
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


In [ ]:
# @title: Creating books data (Update before Run)
BOOK_ID = "fiqh_001-123456"
pages_path = "/content/drive/MyDrive/_AgenticIslam/_Parsed Books Production/Fiqh Zhiali/Fiqh_8_new_pages.json"

# CHANGE THIS
BOOK_PART_NUMBER = 8
BOOK_NAME= "الموسوعة الإسلامية - المجلد 8"

add_refrence_to_embeddings=False

# FAISS settings
isFaiss = False
FAISS_DRIVE_OUTPUT = ""

# Chroma Settings for big files
chroma_collection_name=f"{BOOK_ID}"
chroma_database='E7ya-3loom-eldeen'

print(f"BOOK_ID: {BOOK_ID}")
print(f"BOOK_NAME: {BOOK_NAME}")

BOOK_ID: fiqh_001-123456
BOOK_NAME: الموسوعة الإسلامية - المجلد 8


In [ ]:
# @title Load json page content
file_path = pages_path
with open(file_path, 'r', encoding='utf-8') as f:
          json_pages_data = json.load(f)
pages_list=json_pages_data["pages_list"]
print(pages_list[:4])

[{'page_content': '# الأستاذ الدكتور وهبت الزحيلي\n\n## موسوعة\n\n# الفقه الإسلامي\n\n## والقضايا المعاصرة\n\n### الجزء الثامن\n\n**دار الفكر**\n\n**للساق معرفة المتجددة**\n\n**www.fikr.com**', 'notes': {'footnote': None, 'marginalia': None}, 'page_number': '1'}, {'page_content': '# بسم الله الرحمن الرحيم\n\nيؤتي الحكمة من يشاء ومن يؤت الحكمة فقد أوتي خيرًا كثيرًا وما يذكر إلا أولوا الألباب\n\nموسوعة الفقه الإسلامي والقضايا المعاصرة\n\nالجزء الثامن', 'notes': {'footnote': None, 'marginalia': None}, 'page_number': '2'}, {'page_content': 'موسوعة الفقه الإسلامي والقضايا المعاصرة /\nتأليف وهبة الزحيلي .- دمشق: دار الفكر\n۲۰۱۰ . - ١٣ ج ؛ ٢٥ سم.\nISBN: 978-9933-10-140-4\n۱ - ۲۱۷ ز ح ي م ۲ - العنوان ٣- الزحيلي\nمكتبة الأسد', 'notes': {'footnote': None, 'marginalia': None}, 'page_number': '3'}, {'page_content': '# ثقافة الاختلاف\n\n2012-1433\n\nدار الفكر - دمشق - برامكة\n\n٠٠٩٦٣٤٧٩٧٣٠٠١\n\n٠٠٩٦٣١١٣٠٠١\n\nhttp://www.fikr.com/\ne-mail:fikr@fikr.net\n\nموسوعة الفقه الإسلامي والقضايا المعاصرة\nأ. 

In [ ]:
# @title Pages object for documetns
pages = []
if add_refrence_to_embeddings:
  for p in pages_list:
    if not "page_number" in p or not "page_content" in p:
      continue
    pages.append(
        {
        "page_number":p["page_number"],
        "page_content":f'{p["page_content"]}\n\n --- المصادر \n\n {p["notes"].get("footnote"," ")} \n {p["notes"].get("marginalia"," ")}'
        })
else:
  for p in pages_list:
    if not "page_number" in p or not "page_content" in p:
      continue
    pages.append(
        {
        "page_number":p.get("page_number",-1),
        "page_content":f'{p["page_content"]}'
        })
print(pages[:5])

[{'page_number': '1', 'page_content': '# الأستاذ الدكتور وهبت الزحيلي\n\n## موسوعة\n\n# الفقه الإسلامي\n\n## والقضايا المعاصرة\n\n### الجزء الثامن\n\n**دار الفكر**\n\n**للساق معرفة المتجددة**\n\n**www.fikr.com**'}, {'page_number': '2', 'page_content': '# بسم الله الرحمن الرحيم\n\nيؤتي الحكمة من يشاء ومن يؤت الحكمة فقد أوتي خيرًا كثيرًا وما يذكر إلا أولوا الألباب\n\nموسوعة الفقه الإسلامي والقضايا المعاصرة\n\nالجزء الثامن'}, {'page_number': '3', 'page_content': 'موسوعة الفقه الإسلامي والقضايا المعاصرة /\nتأليف وهبة الزحيلي .- دمشق: دار الفكر\n۲۰۱۰ . - ١٣ ج ؛ ٢٥ سم.\nISBN: 978-9933-10-140-4\n۱ - ۲۱۷ ز ح ي م ۲ - العنوان ٣- الزحيلي\nمكتبة الأسد'}, {'page_number': '4', 'page_content': '# ثقافة الاختلاف\n\n2012-1433\n\nدار الفكر - دمشق - برامكة\n\n٠٠٩٦٣٤٧٩٧٣٠٠١\n\n٠٠٩٦٣١١٣٠٠١\n\nhttp://www.fikr.com/\ne-mail:fikr@fikr.net\n\nموسوعة الفقه الإسلامي والقضايا المعاصرة\nأ. د. وهبة الزحيلي\n\nالجزء الثامن\n\nالرقم الاصطلاحي: ٨-٢٢٤١,٠١١\n\nالرقم الدولي: 4-140-10-9933-978 :ISBN\n\nالتصنيف الموضوعي : ٢

In [ ]:
# ===========================================
# @title Build Documents (element-by-element)
# ===========================================

# @title Convert markdown to normal text
import markdown
from bs4 import BeautifulSoup

def convert_md_to_plain_text(markdown_string):
    """
    Converts a Markdown string to plain text by first converting it to HTML
    and then stripping the HTML tags.
    """
    # Convert Markdown to HTML
    html = markdown.markdown(markdown_string)

    # Use BeautifulSoup to parse the HTML and extract the text
    soup = BeautifulSoup(html, "html.parser")
    plain_text = soup.get_text()

    return plain_text

def generate_documents(
    data : List[Dict[str, Any]],
):

    docs: List[Document] = []

    for item in data:
      doc_meta = {
          "book_id": BOOK_ID,
          "book_name": BOOK_NAME,
          "book_part_number": BOOK_PART_NUMBER,
          "page_number":item["page_number"]
      }

      # Give each chunk a stable ID
      doc_id = f"{BOOK_ID}-{uuid.uuid4().hex}"

      # Build the Document
      docs.append(Document(
          page_content=convert_md_to_plain_text(item["page_content"]),
          metadata=doc_meta,
          id=doc_id
      ))
    return docs

docs = generate_documents(pages)

In [ ]:
print(f"Prepared {len(docs)} document.")
print(docs[160].page_content)
print(docs[160].metadata["page_number"])

Prepared 791 document.
المحرمات من النساء أو الأنكحة المحرمة
وإن كانت النصرانية: فالأظهر حلها للمسلم إن علم دخول قومها، أي آبائها أي أول من تدين منهم في ذلك الدين - أي دين عيسى عليه السلام، قبل نسخه وتحريفه، لتمسكهم بذلك الدين حين كان حقاً. فإن دخلوا فيه بعد التحريف فالأصح المنع، وإن تمسكوا بغير المحرف فتحل في الأظهر.
والراجح لدي هو قول الجمهور، لإطلاق الأدلة القاضية بجواز الزواج بالكتابيات، دون تقييد بشيء.
الزواج بالمجوسيات:
قال أكثر الفقهاء (۱) : ليس المجوس أهل كتاب، للآية المتقدمة ﴿أَن تَقُولُوا إِنَّمَا أُنزِلَ الْكِتَابُ عَلَى طَائِفَتَيْنِ مِن قَبْلِنَا ﴾ [الأنعام: ١٥٦/٦] فأخبر تعالى : أن أهل الكتاب طائفتان، فلو كان المجوس أهل كتاب لكانوا ثلاث طوائف.
وأيضاً إن المجوس لا ينتحلون شيئاً في كتب الله المنزلة على أنبيائه وإنما يقرؤون كتاب زرادشت، وكان متنبياً كذاباً ، فليسوا إذن أهل كتاب.
ويدل له : أن عمر ذكر المجوس بالنسبة لأخذ الجزية منهم، فقال: ما أدري كيف أصنع في أمرهم ؟ فقال له عبد الرحمن بن عوف : أشهد لسمعت رسول الله ﷺ يقول: «سنوا بهم سنة أهل الكتاب» رواه الشافعي، وهو دليل على أن

In [ ]:
# @title Chunking the documetns
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)
print(f"Prepared {len(texts)} chunks.")
print(texts[10])

Prepared 1589 chunks.
page_content='المحتوى
الفصل السابع : حقوق الزواج وواجباته
المبحث الأول - حقوق الزوجة: ٣١٦
المبحث الثاني - حقوق الزوج: ٣١٧
المبحث الثالث - الحقوق المشتركة بين الزوجين: ٣٢٣
الباب الثاني : انحلال الزواج وآثاره
الفصل الأول : الطلاق
الفرق بين الفسخ والطلاق: ٣٣٠
الأول - حقيقة كل منهما: ٣٣٥
الثاني - أسباب كل منهما: ٣٣٦
الثالث - أثر كل منهما: ٣٣٦
المبحث الأول - معنى الطلاق ومشروعيته، وحكمه، وركنه، وحكمته، وسبب جعله بيد الرجل: ٣٣٧
المبحث الثاني - شروط الطلاق وقدره ومحله وصيغته: ٣٤٣
المبحث الثالث - قيود إيقاع الطلاق شرعاً: ٣٥١
أولاً - أن يكون الطلاق لحاجة مقبولة شرعاً وعرفاً: ٣٨٤
ثانياً - أن يكون الطلاق في طهر لم يجامعها فيه: ٣٨٤
ثالثاً - أن يكون الطلاق مفرقاً ليس بأكثر من واحدة: ٣٨٦
المبحث الرابع - التوكيل في الطلاق وتفويضه: ٣٨٩
المبحث الخامس - أنواع الطلاق وحكم كل نوع: ٣٩٧
تقسيم الطلاق من حيث السنة والبدعة: ٤٠٧
تقسيم الطلاق إلى رجعي وبائن: ٤٠٧
تقسيم الطلاق إلى منجز ومعلق ومضاف للمستقبل: ٤١٣
ملحق - حكم طلاق المريض مرض الموت: ٤٢٢
٤٣١' metadata={'book_id': 'fiqh_001-123456',

# Adding Documents to VS



In [ ]:
# @title prepare Embeddings Function
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
# @title Adding in FAISS
if isFaiss:
  vector_store = FAISS.from_documents(texts, embeddings)
  vector_store.save_local(f"{FAISS_DRIVE_OUTPUT}/vectorDB")
else:
  print("NO FAISS")

NO FAISS


In [ ]:
# @title Adding in Chroma
from langchain_chroma import Chroma

if not isFaiss:
  api_key='ck-5ZouSNRMWbN4etnwop1WZ7hatQBt8nZRbeKo1A22kZvJ'
  tenant='26b9b170-8f0f-48b3-ad2a-d9e7a3d94e4c'

  vector_store = Chroma(
      collection_name=chroma_collection_name,
      embedding_function=embeddings,
      chroma_cloud_api_key=api_key,
      tenant=tenant,
      database=chroma_database,
  )
  vector_store
else:
  print("NO CHROMA")

In [ ]:
# @title Delete from Chroma <DANGER>
# import chromadb

# where_filter = {
#     "book_part_number":
# }

# # Delete the records that match the filter
# # vector_store._collection.delete(where=where_filter)

# # To verify the records have been deleted, you can perform a 'get' operation with the same filter
# # and check if the result is empty.
# results = vector_store._collection.get(where=where_filter)
# print("Deleted records:", results)


In [ ]:
# @title Adding in Chroma

if not isFaiss:
  from langchain_chroma import Chroma
  from langchain_community.embeddings import OpenAIEmbeddings
  from langchain_core.documents import Document


  # Define batch size
  batch_size = 150 # Reduced to mitigate timeout errors

  # Loop and add in batches
  for i in range(0, len(texts), batch_size):
      batch = texts[i : i + batch_size]
      print(f"Adding batch {i // batch_size + 1} of size {len(batch)}...")
      vector_store.add_documents(batch)

  print("All documents have been added in batches.")
else:
  print("NO CHROMA")

Adding batch 1 of size 150...
Adding batch 2 of size 150...
Adding batch 3 of size 150...
Adding batch 4 of size 150...
Adding batch 5 of size 150...
Adding batch 6 of size 150...
Adding batch 7 of size 150...
Adding batch 8 of size 150...
Adding batch 9 of size 150...
Adding batch 10 of size 150...
Adding batch 11 of size 89...
All documents have been added in batches.


## Quering the VS

In [ ]:
# vector_store = Chroma(
#       collection_name=chroma_collection_name,
#       embedding_function=embeddings,
#       chroma_cloud_api_key=api_key,
#       tenant=tenant,
#       database=chroma_database,
#   )

query =   "ما حكم تارك الصلاة  "
results = vector_store.similarity_search_with_score(
    query, k=5,
)
for res, score in results:
    print(f"* [SIM={score:3f}]")
    print(f"* [{res.metadata}]")
    print(f"* {res.page_content}")
    print("--"*50)

* [SIM=0.198862]
* [{'book_part_number': 13, 'page_number': '754', 'book_id': 'fiqh_001-123456', 'book_name': 'الموسوعة الإسلامية - المجلد 13'}]
* وهذا دليل على أن ترك الصلاة من موجبات الكفر.
وأميل إلى الرأي الأول، وهو الحكم بعدم كفر تارك الصلاة ونحوها، للأدلة الكثيرة القاطعة بعدم خلود المسلم في النار، بعد النطق
----------------------------------------------------------------------------------------------------
* [SIM=0.233831]
* [{'book_name': 'الموسوعة الإسلامية - المجلد 1', 'book_id': 'fiqh_001-123456', 'book_part_number': 1, 'page_number': 564}]
* وترك الصلاة موجب للعقوبة الأخروية والدنيوية، أما الأخروية فلقوله تعالى : ما سلككُمْ فِي سَفَرَ قَالُوا لَمْ نَكُ مِنَ الْمُصَلِّينَ ﴾ [المدثر: ٤٢/٧٤-٤٣] فَوَيْلٌ لِلْمُصَلِّينَ الَّذِينَ هُمْ عَن صَلَاتِهِمْ سَاهُونَ ﴾ [ الــمــاعــون : ٤/١٠٧-٥]، فَخَلَفَ مِنْ بَعْدِهِمْ خَلْفٌ أَضَاعُوا الصَّلَوٰةَ وَاتَّبَعُوا الشَّهَوَاتِ فَسَوْفَ يَلْقَوْنَ غَيًّا [مــريـــم : ٥٩/١٩]. وقال : من ترك الصلاة متعمداً ، فقد برئت منه ذمة الله ورسوله»(۲).
وأ

In [ ]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5})
retriever.invoke(query)

[Document(id='2fb04522-14cd-4a03-95cb-9a56781fd7a3', metadata={'page_number': '754', 'book_name': 'الموسوعة الإسلامية - المجلد 13', 'book_id': 'fiqh_001-123456', 'book_part_number': 13}, page_content='وهذا دليل على أن ترك الصلاة من موجبات الكفر.\nوأميل إلى الرأي الأول، وهو الحكم بعدم كفر تارك الصلاة ونحوها، للأدلة الكثيرة القاطعة بعدم خلود المسلم في النار، بعد النطق'),
 Document(id='0d93acc4-2ac4-4a29-967e-0213a900b386', metadata={'book_id': 'fiqh_001-123456', 'book_name': 'الموسوعة الإسلامية - المجلد 3', 'page_number': '383', 'book_part_number': 3}, page_content='وإن كانت اليمين على ترك الواجب أو على فعل المعصية كأن قال : ( والله لا'),
 Document(id='c6810161-1dc7-45d8-870d-2c171dc96932', metadata={'book_part_number': 2, 'page_number': '8', 'book_id': 'fiqh_001-123456', 'book_name': 'الموسوعة الإسلامية - المجلد الثاني'}, page_content='المحتوى\nالمطلب الثالث - سجدة الشكر :\nالمبحث الثاني - قضاء الفوائت :\nأولاً - معنى القضاء وحكمه شرعاً :\nثانياً - أعذار سقوط الصلاة وتأخيرها :\nأ - أعذا

#### Cohere Reranker

In [ ]:
from langchain.retrievers.document_compressors import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever
def getResultReRanked(number, db):
    cohereKey = "REPLACED_COHERE_KEY"
    compressor = CohereRerank(cohere_api_key=cohereKey, model="rerank-multilingual-v3.0",
                              top_n=number)
    compression_retriever = ContextualCompressionRetriever(base_compressor=compressor,
                                                           base_retriever=db.as_retriever(
                                                               search_kwargs={"k": number}))
    return compression_retriever

retriever = getResultReRanked(5, vector_store)
retriever.invoke(query)